In [1]:
import pandas as pd
from retail.snapshots import build_snapshot
from retail.evaluate import log_results
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

In [2]:
purchase_df = pd.read_parquet("../data/processed/transaction_data.parquet")
return_df = pd.read_parquet("../data/processed/returns_data.parquet")

Building one snapshot at 2011-03-01

In [3]:
df = build_snapshot(purchase_df, return_df, cut_off='2010-07-01')

In [4]:
df.head()

,Customer ID,product_breadth,frequency_180d,monetary_180d,country_count_180d,aov_180d,tenure_days,recency_days,uk_invoice_share_180d,monetary_last_90d,monetary_first_90d,spend_trend,return_amount,return_rate,churned,snapshot_date
0,12346,25,6,259.36,1,43.226667,198,2,1.0,142.31,117.05,25.26,-103.5,0.399059,1,2010-07-01
1,12349,46,1,1068.52,1,1068.520000,62,62,0.0,1068.52,0.00,1068.52,0.0,0.000000,1,2010-07-01
2,12355,22,1,488.21,1,488.210000,40,40,0.0,488.21,0.00,488.21,0.0,0.000000,1,2010-07-01
3,12358,17,1,268.10,1,268.100000,204,23,0.0,268.10,0.00,268.10,0.0,0.000000,1,2010-07-01
4,12359,50,3,1173.14,1,391.046667,207,8,0.0,489.80,683.34,-193.54,-15.8,0.013468,1,2010-07-01


In [5]:
df.columns

Index(['Customer ID', 'product_breadth', 'frequency_180d', 'monetary_180d',
       'country_count_180d', 'aov_180d', 'tenure_days', 'recency_days',
       'uk_invoice_share_180d', 'monetary_last_90d', 'monetary_first_90d',
       'spend_trend', 'return_amount', 'return_rate', 'churned',
       'snapshot_date'],
      dtype='str')

In [6]:
df.shape

(2748, 16)

In [7]:
df['Customer ID'].nunique() #customers = 2748

2748

In [8]:
df['churned'].value_counts(normalize=True)

churned
0    0.507642
1    0.492358
Name: proportion, dtype: float64

### Baselines, then the model

In [9]:
X = df.drop(columns='churned')
y = df['churned']

In [10]:
X_train, X_test, y_train, y_test =  train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [11]:
X_train.shape

(2198, 15)

In [12]:
def model_train_predict(model, X_train, y_train, X_test, y_test, model_name='Model'):
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    metrics = {
        "model_name": model_name,
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "log_loss": log_loss(y_test, y_proba),        
    }

    return metrics

In [13]:
dummy_clf = DummyClassifier(strategy='prior')
dummy_metrics = model_train_predict(dummy_clf, X_train, y_train, X_test, y_test, model_name='Dummy Classifier')

log_results(dummy_metrics)

In [14]:
dummy_metrics

{'model_name': 'Dummy Classifier',
 'roc_auc': 0.5,
 'pr_auc': 0.49272727272727274,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'log_loss': 0.693041817909472}

In [15]:
logreg_recency= LogisticRegression(max_iter=1000)
logreg_recency_metrics = model_train_predict(logreg_recency, 
                                            X_train[['recency_days']], 
                                            y_train, 
                                            X_test[['recency_days']], 
                                            y_test, 
                                            model_name='Logistic Regression - Recency')

log_results(logreg_recency_metrics)

In [16]:
logreg_recency_metrics

{'model_name': 'Logistic Regression - Recency',
 'roc_auc': 0.6695631472443755,
 'pr_auc': 0.6424253003771558,
 'precision': 0.6528925619834711,
 'recall': 0.5830258302583026,
 'f1': 0.6159844054580896,
 'log_loss': 0.6501655470322517}

In [48]:
featured_column = ['product_breadth', 'frequency_180d', 'monetary_180d',
       'country_count_180d', 'aov_180d', 'tenure_days', 'recency_days',
       'uk_invoice_share_180d', 'monetary_last_90d', 'monetary_first_90d',
       'spend_trend', 'return_amount', 'return_rate']

In [ ]:
logreg_all = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)
logreg_all_metrics = model_train_predict(logreg_all,
                    X_train[featured_column],
                    y_train,
                    X_test[featured_column],
                    y_test,
                    model_name='Logistic Regression - All Features')

log_results(logreg_all_metrics)


In [18]:
logreg_all_metrics

{'model_name': 'Logistic Regression - All Features',
 'roc_auc': 0.7380073800738008,
 'pr_auc': 0.6750188560631887,
 'precision': 0.6666666666666666,
 'recall': 0.7527675276752768,
 'f1': 0.707105719237435,
 'log_loss': 0.6074487949361752}

Clearly the roc auc score has increased and log loss decreased when adding more additional features incomparison to just using recency

In [19]:

skewed_features = [
    "monetary_180d",
    "frequency_180d",
    "aov_180d",
]

X_train_log = X_train.copy()
X_test_log = X_test.copy()

X_train_log[skewed_features] = np.log1p(
    X_train_log[skewed_features]
)

X_test_log[skewed_features] = np.log1p(
    X_test_log[skewed_features]
)

In [ ]:
logreg_log = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg_log_metrics = model_train_predict(logreg_log,
                    X_train_log[featured_column],
                    y_train,
                    X_test_log[featured_column],
                    y_test,
                    model_name='Logistic Regression - All Features(np log)')

log_results(logreg_log_metrics)

In [21]:
logreg_log_metrics

{'model_name': 'Logistic Regression - All Features(np log)',
 'roc_auc': 0.7424248436032748,
 'pr_auc': 0.6700604779095154,
 'precision': 0.6597938144329897,
 'recall': 0.7084870848708487,
 'f1': 0.6832740213523132,
 'log_loss': 0.5991723693632244}

Clearly np.log1p() reduces right skew in monetary_180d, frequency_180d, and aov_180d. and improves the model AUC and reduces logloss

Intepret the coefficients

First model just used recency

In [37]:
recency_coeff = logreg_recency.coef_[0][0]

In [39]:

coef_df = pd.DataFrame({
    "feature": ['recency_days'],
    "coefficient": [recency_coeff],
})

coef_df.head()

coef_df["odds_ratio"] = np.exp(coef_df["coefficient"])

coef_df.sort_values("coefficient", ascending=False)

,feature,coefficient,odds_ratio
0,recency_days,0.012021,1.012093


For every additional day since the customer's last purchase, the odds of churn are multiplied by approximately 1.012, holding all other features constant. \[
(1.012093-1)X 100 ~ 1.21%


In [49]:
all_model_coef = logreg_all['logisticregression'].coef_[0]

coef_df = pd.DataFrame({
    "feature": featured_column,
    "coefficient": all_model_coef,
    
})

coef_df["odds_ratio"] = np.exp(coef_df["coefficient"])

coef_df.sort_values("coefficient", ascending=False)

,feature,coefficient,odds_ratio
6,recency_days,0.279905,1.323004
4,aov_180d,0.065361,1.067544
7,uk_invoice_share_180d,-0.007177,0.992849
12,return_rate,-0.025215,0.975100
10,spend_trend,-0.151636,0.859301
3,country_count_180d,-0.170706,0.843069
0,product_breadth,-0.288163,0.749640
5,tenure_days,-0.304250,0.737677
9,monetary_first_90d,-0.314056,0.730478
11,return_amount,-0.340481,0.711428


keeping all the other features constant, for every unit increase in the standard deviation in  number of days since customer's last purchase multiplies teh odds of churn by  1.32 ~ 32.3% higher odds of churn


In [50]:
all_model_log_coef = logreg_log['logisticregression'].coef_[0]

coef_df = pd.DataFrame({
    "feature": featured_column,
    "coefficient": all_model_log_coef,
    
})

coef_df["odds_ratio"] = np.exp(coef_df["coefficient"])

coef_df.sort_values("coefficient", ascending=False)

,feature,coefficient,odds_ratio
6,recency_days,0.209939,1.233603
9,monetary_first_90d,0.053354,1.054804
12,return_rate,-0.010390,0.989664
8,monetary_last_90d,-0.028843,0.971569
0,product_breadth,-0.032019,0.968488
7,uk_invoice_share_180d,-0.039057,0.961696
2,monetary_180d,-0.052088,0.949246
10,spend_trend,-0.122930,0.884326
11,return_amount,-0.140200,0.869185
3,country_count_180d,-0.166669,0.846480


Keepign all the features constanst, for every unit increase in the standard deviation in log normalised monetary_180d mutliplies the odds of churn by 0.94 corresponding to a 4.88% decrease in the odds of churn.
=(0.9512-1)*100 


Formula (odd ratio -1)*100
